# 07. Spark SQL, Temporary Views & ANSI SQL Interoperability

One of PySpark's greatest strengths is that you can switch effortlessly between Python DataFrame code and standard ANSI SQL!

### Topics Covered:
1. **What is Spark SQL?**
2. **Creating Temporary Views (`createOrReplaceTempView` vs `createGlobalTempView`)**
3. **Querying DataFrames using `spark.sql(...)`**
4. **Writing Complex SQL: CTEs (WITH clause), Window Functions, Subqueries**
5. **Seamless Interoperability: Python to SQL and SQL back to Python**
6. **Inspecting and Managing the Catalog (`spark.catalog`)**
7. **Stopping the SparkSession to free memory**


---
## 1. Initializing SparkSession


In [ ]:


from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("07_Spark_SQL") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession ready!")


---
## 2. Preparing Data & Registering Temporary Views

To run SQL queries directly on a DataFrame, you first register it as a **Temporary View**.

* `createOrReplaceTempView("view_name")`:
  * Session-scoped: Available only within the current `SparkSession`.
  * If the session ends, the view disappears.
* `createGlobalTempView("view_name")`:
  * Cluster-scoped: Shared across different SparkSessions within the same cluster.
  * Accessed using the `global_temp` database prefix: `SELECT * FROM global_temp.view_name`.


In [ ]:
# Sample transactions data
transactions = [
    (1, "Alice", "Electronics", 1200.0, "2023-01-10"),
    (2, "Bob", "Clothing", 80.0, "2023-01-12"),
    (3, "Alice", "Groceries", 65.0, "2023-01-15"),
    (4, "Charlie", "Electronics", 450.0, "2023-01-20"),
    (5, "Bob", "Electronics", 300.0, "2023-01-22"),
    (6, "Diana", "Clothing", 150.0, "2023-01-25"),
    (7, "Alice", "Clothing", 220.0, "2023-01-28")
]

columns = ["txn_id", "customer", "category", "amount", "txn_date"]
df = spark.createDataFrame(transactions, columns)

# Register as a local temporary view
df.createOrReplaceTempView("txns")

print("Temporary view 'txns' created successfully!")


---
## 3. Querying with `spark.sql()`

`spark.sql(...)` runs standard ANSI SQL queries and returns a PySpark DataFrame.
This means you can chain Python methods directly on the SQL output!


In [ ]:
# Basic SQL query
result = spark.sql("""
    SELECT 
        customer,
        category,
        amount
    FROM txns
    WHERE amount >= 100
    ORDER BY amount DESC
""")

result.show()


In [ ]:
# SQL Aggregations: Total and Average spend per customer
agg_sql = spark.sql("""
    SELECT 
        customer,
        COUNT(txn_id) AS total_orders,
        ROUND(SUM(amount), 2) AS total_spend,
        ROUND(AVG(amount), 2) AS avg_spend
    FROM txns
    GROUP BY customer
    HAVING total_spend > 200
    ORDER BY total_spend DESC
""")

agg_sql.show()


---
## 4. Advanced SQL: Common Table Expressions (CTEs) & Window Functions

You can write full modern SQL, including `WITH` clauses and `OVER (PARTITION BY ... ORDER BY ...)`.


In [ ]:
# Using CTE (WITH clause) and Window Function in SQL
window_sql = spark.sql("""
    WITH ranked_purchases AS (
        SELECT 
            customer,
            category,
            amount,
            txn_date,
            DENSE_RANK() OVER (PARTITION BY customer ORDER BY amount DESC) as rank_in_spend
        FROM txns
    )
    SELECT 
        customer,
        category,
        amount,
        txn_date
    FROM ranked_purchases
    WHERE rank_in_spend = 1
    ORDER BY amount DESC
""")

print("Each customer's single most expensive purchase:")
window_sql.show()


---
## 5. Seamless Interoperability: Python + SQL Together

You don't have to choose between Python and SQL. You can mix them freely!


In [ ]:
# Step 1: SQL query creates a DataFrame
filtered_sql_df = spark.sql("""
    SELECT customer, category, amount 
    FROM txns 
    WHERE category = 'Electronics'
""")

# Step 2: Use Python DataFrame API to add new calculated columns
from pyspark.sql.functions import col

enriched_df = filtered_sql_df.withColumn("tax_amount", col("amount") * 0.08) \
                             .withColumn("total_with_tax", col("amount") * 1.08)

# Step 3: Register the enriched DataFrame as a new view
enriched_df.createOrReplaceTempView("electronics_with_tax")

# Step 4: Query the new view with SQL again!
spark.sql("SELECT * FROM electronics_with_tax ORDER BY total_with_tax DESC").show()


---
## 6. Inspecting the Spark Catalog

The `spark.catalog` allows you to inspect registered tables, views, databases, and cached status.


In [ ]:
# List all tables and views in current session
print("Registered Views in Catalog:")
for table in spark.catalog.listTables():
    print(f"  - Table Name: {table.name}, Is Temporary: {table.isTemporary}")

# Drop a temporary view when finished
spark.catalog.dropTempView("electronics_with_tax")
print("Dropped 'electronics_with_tax' view. Remaining views:")
for table in spark.catalog.listTables():
    print(f"  - {table.name}")


---
## 7. Summary of Notebook 07
* `createOrReplaceTempView()` registers any DataFrame as a queryable SQL table.
* `spark.sql("SELECT ...")` runs standard SQL and returns a DataFrame.
* You can write CTEs, window functions, and subqueries directly in Spark SQL.
* You can seamlessly alternate between Python DataFrame transformations and SQL queries.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes before moving to the next notebook.


In [ ]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")
